In [1]:
import tkinter as tk
from tkinter import ttk, messagebox, filedialog
from tkinter.scrolledtext import ScrolledText
import pandas as pd
import joblib
from datetime import datetime

MODEL_FILE = "cardio_best_lightgbm_model.pkl"

try:
    model = joblib.load(MODEL_FILE)
except Exception:
    model = None

root = tk.Tk()
root.title("🏥 Cardiovascular Disease Prediction System")
root.geometry("1200x820")
root.configure(bg="#EAF6FB")

style = ttk.Style()
style.theme_use("clam")
style.configure("TButton", font=("Segoe UI",10,"bold"), padding=6)
style.configure("TLabel", background="#EAF6FB", font=("Segoe UI",10))
style.configure("Header.TLabel", font=("Segoe UI",20,"bold"),
                background="#0077B6", foreground="white")

banner = tk.Frame(root,bg="#0077B6",height=70)
banner.pack(fill="x")
ttk.Label(banner,text="❤ Cardiovascular Disease Prediction Prototype",
          style="Header.TLabel").pack(pady=18)

canvas=tk.Canvas(root,bg="#EAF6FB",highlightthickness=0)
scroll=ttk.Scrollbar(root,orient="vertical",command=canvas.yview)
frame=tk.Frame(canvas,bg="#EAF6FB")
frame.bind("<Configure>",lambda e:canvas.configure(scrollregion=canvas.bbox("all")))
canvas.create_window((0,0),window=frame,anchor="nw")
canvas.configure(yscrollcommand=scroll.set)
canvas.pack(side="left",fill="both",expand=True)
scroll.pack(side="right",fill="y")

fields={}
def entry(parent,text,row):
    ttk.Label(parent,text=text).grid(row=row,column=0,padx=5,pady=5,sticky="w")
    e=ttk.Entry(parent,width=28)
    e.grid(row=row,column=1,padx=5,pady=5)
    fields[text]=e

def combo(parent,text,row,vals):
    ttk.Label(parent,text=text).grid(row=row,column=0,padx=5,pady=5,sticky="w")
    c=ttk.Combobox(parent,values=vals,state="readonly",width=25)
    c.current(0)
    c.grid(row=row,column=1,padx=5,pady=5)
    fields[text]=c

patient=ttk.LabelFrame(frame,text="Patient Information")
patient.pack(fill="x",padx=15,pady=10)

entry(patient,"ID",0)
entry(patient,"Age (Years)",1)
combo(patient,"Gender",2,["1 - Male","2 - Female"])
entry(patient,"Height (cm)",3)
entry(patient,"Weight (kg)",4)
entry(patient,"Systolic BP",5)
entry(patient,"Diastolic BP",6)
combo(patient,"Cholesterol",7,["1 - Normal","2 - Above Normal","3 - Well Above"])
combo(patient,"Glucose",8,["1 - Normal","2 - Above Normal","3 - Well Above"])
combo(patient,"Smoking",9,["0 - No","1 - Yes"])
combo(patient,"Alcohol",10,["0 - No","1 - Yes"])
combo(patient,"Physical Activity",11,["0 - No","1 - Yes"])

pred=tk.StringVar()
conf=tk.StringVar()
risk=tk.StringVar()
bmi_var=tk.StringVar()

history=[]

tree=None

def predict():
    if model is None:
        messagebox.showerror("Error","Model file not found.")
        return
    try:
        age_years=float(fields["Age (Years)"].get())
        age_days=age_years*365
        h=float(fields["Height (cm)"].get())
        w=float(fields["Weight (kg)"].get())
        bmi=w/((h/100)**2)
        bmi_var.set(f"{bmi:.2f}")

        vals=[[float(fields["ID"].get()),
               age_days,
               int(fields["Gender"].get()[0]),
               h,w,
               float(fields["Systolic BP"].get()),
               float(fields["Diastolic BP"].get()),
               int(fields["Cholesterol"].get()[0]),
               int(fields["Glucose"].get()[0]),
               int(fields["Smoking"].get()[0]),
               int(fields["Alcohol"].get()[0]),
               int(fields["Physical Activity"].get()[0]),
               age_years,
               bmi]]

        cols=["id","age","gender","height","weight","ap_hi","ap_lo",
              "cholesterol","gluc","smoke","alco","active","age_years","BMI"]

        X=pd.DataFrame(vals,columns=cols)

        p=model.predict(X)[0]
        pr=model.predict_proba(X)[0].max()*100

        label="Cardiovascular Disease" if p==1 else "No Cardiovascular Disease"
        pred.set(label)
        conf.set(f"{pr:.2f}%")

        if pr>=80:
            risk.set("High")
        elif pr>=60:
            risk.set("Moderate")
        else:
            risk.set("Low")

        tree.insert("",tk.END,values=(datetime.now().strftime("%Y-%m-%d %H:%M"),
                                      label,f"{pr:.2f}%"))
    except Exception as e:
        messagebox.showerror("Error",str(e))

def reset():
    for w in fields.values():
        if isinstance(w,ttk.Combobox):
            w.current(0)
        else:
            w.delete(0,tk.END)
    pred.set("")
    conf.set("")
    risk.set("")
    bmi_var.set("")
    comments.delete("1.0",tk.END)

summary=ttk.LabelFrame(frame,text="Prediction")
summary.pack(fill="x",padx=15,pady=10)

ttk.Button(summary,text="🔍 Predict",command=predict).grid(row=0,column=0,padx=5,pady=5)
ttk.Button(summary,text="♻ Reset",command=reset).grid(row=0,column=1,padx=5,pady=5)

labels=[("Prediction",pred),("Confidence",conf),("Risk",risk),("BMI",bmi_var)]
for i,(t,v) in enumerate(labels,1):
    ttk.Label(summary,text=t).grid(row=i,column=0,sticky="w")
    ttk.Label(summary,textvariable=v,font=("Segoe UI",10,"bold")).grid(row=i,column=1,sticky="w")

fb=ttk.LabelFrame(frame,text="Human Feedback")
fb.pack(fill="x",padx=15,pady=10)

ttk.Combobox(fb,values=["Excellent","Good","Average","Poor"],
             state="readonly").pack(fill="x",padx=5,pady=5)

comments=ScrolledText(fb,height=5)
comments.pack(fill="x",padx=5,pady=5)

treeframe=ttk.LabelFrame(frame,text="Prediction History")
treeframe.pack(fill="both",expand=True,padx=15,pady=10)

tree=ttk.Treeview(treeframe,columns=("Time","Prediction","Confidence"),
                  show="headings",height=10)
for c in ("Time","Prediction","Confidence"):
    tree.heading(c,text=c)
    tree.column(c,width=250)
tree.pack(fill="both",expand=True)

def export_csv():
    f=filedialog.asksaveasfilename(defaultextension=".csv")
    if not f:
        return
    rows=[tree.item(i)["values"] for i in tree.get_children()]
    pd.DataFrame(rows,columns=["Time","Prediction","Confidence"]).to_csv(f,index=False)
    messagebox.showinfo("Done","History exported successfully.")

menubar=tk.Menu(root)
fm=tk.Menu(menubar,tearoff=0)
fm.add_command(label="Export History",command=export_csv)
fm.add_separator()
fm.add_command(label="Exit",command=root.destroy)
menubar.add_cascade(label="File",menu=fm)

hm=tk.Menu(menubar,tearoff=0)
hm.add_command(label="About",
               command=lambda:messagebox.showinfo(
                   "About",
                   "Professional Cardiovascular Disease Prediction Prototype\nLightGBM Model"))
menubar.add_cascade(label="Help",menu=hm)
root.config(menu=menubar)

status=tk.Label(root,text="Ready",bg="#0077B6",fg="white",anchor="w")
status.pack(fill="x",side="bottom")

root.mainloop()
